In [2]:

"""
NewLine Cinema Management System - Server
A socket-based server that manages cinema database operations.
"""

import socket
import threading
import json
import sqlite3
import logging
from datetime import datetime

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class CinemaServer:
    def __init__(self, host='localhost', port=12345):
        self.host = host
        self.port = port
        self.db_name = 'cinema.db'
        self.init_database()
        
    def init_database(self):
        """Initialize the SQLite database with tables and sample data."""
        try:
            conn = sqlite3.connect(self.db_name)
            cursor = conn.cursor()
            
            # Create movies table
            cursor.execute('''
                CREATE TABLE IF NOT EXISTS movies (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    title TEXT NOT NULL,
                    cinema_room INTEGER NOT NULL CHECK (cinema_room BETWEEN 1 AND 7),
                    release_date TEXT NOT NULL,
                    end_date TEXT NOT NULL,
                    tickets_available INTEGER NOT NULL CHECK (tickets_available >= 0),
                    ticket_price REAL NOT NULL CHECK (ticket_price > 0)
                )
            ''')
            
            # Create sales table
            cursor.execute('''
                CREATE TABLE IF NOT EXISTS sales (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    movie_id INTEGER NOT NULL,
                    customer_name TEXT NOT NULL,
                    number_of_tickets INTEGER NOT NULL CHECK (number_of_tickets > 0),
                    total REAL NOT NULL CHECK (total > 0),
                    sale_date TEXT NOT NULL,
                    FOREIGN KEY (movie_id) REFERENCES movies (id)
                )
            ''')
            
            # Check if movies table is empty and populate with sample data
            cursor.execute("SELECT COUNT(*) FROM movies")
            if cursor.fetchone()[0] == 0:
                sample_movies = [
                    ("The Matrix Resurrections", 1, "2024-01-15", "2024-03-15", 150, 12.50),
                    ("Spider-Man: No Way Home", 2, "2024-01-10", "2024-03-10", 200, 15.00),
                    ("Dune: Part Two", 3, "2024-02-01", "2024-04-01", 180, 14.00),
                    ("Avatar: The Way of Water", 4, "2024-01-20", "2024-03-20", 220, 16.50),
                    ("Top Gun: Maverick", 5, "2024-02-05", "2024-04-05", 175, 13.75),
                    ("Black Panther: Wakanda Forever", 6, "2024-01-25", "2024-03-25", 190, 14.25),
                    ("Everything Everywhere All at Once", 7, "2024-02-10", "2024-04-10", 160, 12.00)
                ]
                
                cursor.executemany('''
                    INSERT INTO movies (title, cinema_room, release_date, end_date, tickets_available, ticket_price)
                    VALUES (?, ?, ?, ?, ?, ?)
                ''', sample_movies)
                
                logger.info("Sample movies added to database")
            
            conn.commit()
            conn.close()
            logger.info("Database initialized successfully")
            
        except sqlite3.Error as e:
            logger.error(f"Database initialization error: {e}")
            raise
    
    def get_movies(self):
        """Retrieve all movies from the database."""
        try:
            conn = sqlite3.connect(self.db_name)
            cursor = conn.cursor()
            cursor.execute('''
                SELECT id, title, cinema_room, release_date, end_date, tickets_available, ticket_price
                FROM movies ORDER BY title
            ''')
            movies = cursor.fetchall()
            conn.close()
            
            # Convert to list of dictionaries
            movie_list = []
            for movie in movies:
                movie_list.append({
                    'id': movie[0],
                    'title': movie[1],
                    'cinema_room': movie[2],
                    'release_date': movie[3],
                    'end_date': movie[4],
                    'tickets_available': movie[5],
                    'ticket_price': movie[6]
                })
            
            return {'status': 'success', 'data': movie_list}
            
        except sqlite3.Error as e:
            logger.error(f"Error retrieving movies: {e}")
            return {'status': 'error', 'message': f"Database error: {e}"}
    
    def add_movie(self, movie_data):
        """Add a new movie to the database."""
        try:
            required_fields = ['title', 'cinema_room', 'release_date', 'end_date', 'tickets_available', 'ticket_price']
            for field in required_fields:
                if field not in movie_data:
                    return {'status': 'error', 'message': f"Missing required field: {field}"}
            
            # Validate cinema room
            if not (1 <= movie_data['cinema_room'] <= 7):
                return {'status': 'error', 'message': "Cinema room must be between 1 and 7"}
            
            # Validate tickets and price
            if movie_data['tickets_available'] < 0:
                return {'status': 'error', 'message': "Tickets available cannot be negative"}
            
            if movie_data['ticket_price'] <= 0:
                return {'status': 'error', 'message': "Ticket price must be positive"}
            
            conn = sqlite3.connect(self.db_name)
            cursor = conn.cursor()
            
            # Check if cinema room is already occupied
            cursor.execute("SELECT id FROM movies WHERE cinema_room = ?", (movie_data['cinema_room'],))
            if cursor.fetchone():
                conn.close()
                return {'status': 'error', 'message': f"Cinema room {movie_data['cinema_room']} is already occupied"}
            
            cursor.execute('''
                INSERT INTO movies (title, cinema_room, release_date, end_date, tickets_available, ticket_price)
                VALUES (?, ?, ?, ?, ?, ?)
            ''', (movie_data['title'], movie_data['cinema_room'], movie_data['release_date'],
                  movie_data['end_date'], movie_data['tickets_available'], movie_data['ticket_price']))
            
            movie_id = cursor.lastrowid
            conn.commit()
            conn.close()
            
            logger.info(f"Movie added successfully with ID: {movie_id}")
            return {'status': 'success', 'message': 'Movie added successfully', 'movie_id': movie_id}
            
        except sqlite3.Error as e:
            logger.error(f"Error adding movie: {e}")
            return {'status': 'error', 'message': f"Database error: {e}"}
    
    def update_movie(self, movie_data):
        """Update an existing movie in the database."""
        try:
            if 'id' not in movie_data:
                return {'status': 'error', 'message': "Movie ID is required for update"}
            
            conn = sqlite3.connect(self.db_name)
            cursor = conn.cursor()
            
            # Check if movie exists
            cursor.execute("SELECT id FROM movies WHERE id = ?", (movie_data['id'],))
            if not cursor.fetchone():
                conn.close()
                return {'status': 'error', 'message': "Movie not found"}
            
            # Build update query dynamically
            update_fields = []
            values = []
            
            updatable_fields = ['title', 'cinema_room', 'release_date', 'end_date', 'tickets_available', 'ticket_price']
            for field in updatable_fields:
                if field in movie_data:
                    if field == 'cinema_room' and not (1 <= movie_data[field] <= 7):
                        conn.close()
                        return {'status': 'error', 'message': "Cinema room must be between 1 and 7"}
                    
                    if field == 'tickets_available' and movie_data[field] < 0:
                        conn.close()
                        return {'status': 'error', 'message': "Tickets available cannot be negative"}
                    
                    if field == 'ticket_price' and movie_data[field] <= 0:
                        conn.close()
                        return {'status': 'error', 'message': "Ticket price must be positive"}
                    
                    # Check cinema room availability (if being updated)
                    if field == 'cinema_room':
                        cursor.execute("SELECT id FROM movies WHERE cinema_room = ? AND id != ?", 
                                     (movie_data[field], movie_data['id']))
                        if cursor.fetchone():
                            conn.close()
                            return {'status': 'error', 'message': f"Cinema room {movie_data[field]} is already occupied"}
                    
                    update_fields.append(f"{field} = ?")
                    values.append(movie_data[field])
            
            if not update_fields:
                conn.close()
                return {'status': 'error', 'message': "No valid fields to update"}
            
            values.append(movie_data['id'])
            query = f"UPDATE movies SET {', '.join(update_fields)} WHERE id = ?"
            
            cursor.execute(query, values)
            conn.commit()
            conn.close()
            
            logger.info(f"Movie {movie_data['id']} updated successfully")
            return {'status': 'success', 'message': 'Movie updated successfully'}
            
        except sqlite3.Error as e:
            logger.error(f"Error updating movie: {e}")
            return {'status': 'error', 'message': f"Database error: {e}"}
    
    def delete_movie(self, movie_id):
        """Delete a movie from the database."""
        try:
            conn = sqlite3.connect(self.db_name)
            cursor = conn.cursor()
            
            # Check if movie exists
            cursor.execute("SELECT id FROM movies WHERE id = ?", (movie_id,))
            if not cursor.fetchone():
                conn.close()
                return {'status': 'error', 'message': "Movie not found"}
            
            # Delete related sales first
            cursor.execute("DELETE FROM sales WHERE movie_id = ?", (movie_id,))
            
            # Delete the movie
            cursor.execute("DELETE FROM movies WHERE id = ?", (movie_id,))
            
            conn.commit()
            conn.close()
            
            logger.info(f"Movie {movie_id} deleted successfully")
            return {'status': 'success', 'message': 'Movie deleted successfully'}
            
        except sqlite3.Error as e:
            logger.error(f"Error deleting movie: {e}")
            return {'status': 'error', 'message': f"Database error: {e}"}
    
    def update_tickets(self, movie_id, new_ticket_count):
        """Update the number of available tickets for a movie."""
        try:
            if new_ticket_count < 0:
                return {'status': 'error', 'message': "Ticket count cannot be negative"}
            
            conn = sqlite3.connect(self.db_name)
            cursor = conn.cursor()
            
            # Check if movie exists
            cursor.execute("SELECT id FROM movies WHERE id = ?", (movie_id,))
            if not cursor.fetchone():
                conn.close()
                return {'status': 'error', 'message': "Movie not found"}
            
            cursor.execute("UPDATE movies SET tickets_available = ? WHERE id = ?", 
                         (new_ticket_count, movie_id))
            
            conn.commit()
            conn.close()
            
            logger.info(f"Tickets updated for movie {movie_id}: {new_ticket_count}")
            return {'status': 'success', 'message': 'Tickets updated successfully'}
            
        except sqlite3.Error as e:
            logger.error(f"Error updating tickets: {e}")
            return {'status': 'error', 'message': f"Database error: {e}"}
    
    def record_sale(self, sale_data):
        """Record a ticket sale and update available tickets."""
        try:
            required_fields = ['movie_id', 'customer_name', 'number_of_tickets']
            for field in required_fields:
                if field not in sale_data:
                    return {'status': 'error', 'message': f"Missing required field: {field}"}
            
            if sale_data['number_of_tickets'] <= 0:
                return {'status': 'error', 'message': "Number of tickets must be positive"}
            
            conn = sqlite3.connect(self.db_name)
            cursor = conn.cursor()
            
            # Get movie details and check availability
            cursor.execute("SELECT tickets_available, ticket_price, title FROM movies WHERE id = ?", 
                         (sale_data['movie_id'],))
            movie_info = cursor.fetchone()
            
            if not movie_info:
                conn.close()
                return {'status': 'error', 'message': "Movie not found"}
            
            available_tickets, price, title = movie_info
            
            if available_tickets < sale_data['number_of_tickets']:
                conn.close()
                return {'status': 'error', 'message': f"Only {available_tickets} tickets available"}
            
            # Calculate total
            total = price * sale_data['number_of_tickets']
            sale_date = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            
            # Record the sale
            cursor.execute('''
                INSERT INTO sales (movie_id, customer_name, number_of_tickets, total, sale_date)
                VALUES (?, ?, ?, ?, ?)
            ''', (sale_data['movie_id'], sale_data['customer_name'], 
                  sale_data['number_of_tickets'], total, sale_date))
            
            sale_id = cursor.lastrowid
            
            # Update available tickets
            new_available = available_tickets - sale_data['number_of_tickets']
            cursor.execute("UPDATE movies SET tickets_available = ? WHERE id = ?", 
                         (new_available, sale_data['movie_id']))
            
            conn.commit()
            conn.close()
            
            logger.info(f"Sale recorded: ID {sale_id}, Movie: {title}, Customer: {sale_data['customer_name']}")
            
            return {
                'status': 'success',
                'message': 'Sale recorded successfully',
                'sale_id': sale_id,
                'total': total,
                'movie_title': title,
                'remaining_tickets': new_available
            }
            
        except sqlite3.Error as e:
            logger.error(f"Error recording sale: {e}")
            return {'status': 'error', 'message': f"Database error: {e}"}
    
    def handle_client(self, client_socket, address):
        """Handle individual client connections."""
        logger.info(f"Client connected from {address}")
        
        try:
            while True:
                # Receive data from client
                data = client_socket.recv(4096).decode('utf-8')
                if not data:
                    break
                
                try:
                    request = json.loads(data)
                    logger.info(f"Received request: {request.get('action', 'unknown')}")
                    
                    # Process the request based on action
                    action = request.get('action')
                    
                    if action == 'get_movies':
                        response = self.get_movies()
                    elif action == 'add_movie':
                        response = self.add_movie(request.get('data', {}))
                    elif action == 'update_movie':
                        response = self.update_movie(request.get('data', {}))
                    elif action == 'delete_movie':
                        response = self.delete_movie(request.get('movie_id'))
                    elif action == 'update_tickets':
                        response = self.update_tickets(request.get('movie_id'), request.get('ticket_count'))
                    elif action == 'record_sale':
                        response = self.record_sale(request.get('data', {}))
                    else:
                        response = {'status': 'error', 'message': 'Unknown action'}
                    
                    # Send response back to client
                    response_json = json.dumps(response)
                    client_socket.send(response_json.encode('utf-8'))
                    
                except json.JSONDecodeError:
                    error_response = {'status': 'error', 'message': 'Invalid JSON format'}
                    client_socket.send(json.dumps(error_response).encode('utf-8'))
                except Exception as e:
                    logger.error(f"Error processing request: {e}")
                    error_response = {'status': 'error', 'message': f'Server error: {str(e)}'}
                    client_socket.send(json.dumps(error_response).encode('utf-8'))
                    
        except Exception as e:
            logger.error(f"Client handler error: {e}")
        finally:
            client_socket.close()
            logger.info(f"Client {address} disconnected")
    
    def start_server(self):
        """Start the server and listen for client connections."""
        try:
            server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            server_socket.bind((self.host, self.port))
            server_socket.listen(5)
            
            logger.info(f"Cinema server started on {self.host}:{self.port}")
            print(f"NewLine Cinema Server running on {self.host}:{self.port}")
            print("Waiting for client connections...")
            
            while True:
                client_socket, address = server_socket.accept()
                client_thread = threading.Thread(
                    target=self.handle_client,
                    args=(client_socket, address)
                )
                client_thread.daemon = True
                client_thread.start()
                
        except Exception as e:
            logger.error(f"Server error: {e}")
            print(f"Server error: {e}")
        finally:
            if 'server_socket' in locals():
                server_socket.close()

if __name__ == "__main__":
    server = CinemaServer()
    try:
        server.start_server()
    except KeyboardInterrupt:
        print("\nServer shutting down...")
        logger.info("Server shutdown requested")

2025-06-13 05:07:13,443 - INFO - Database initialized successfully
2025-06-13 05:07:13,450 - ERROR - Server error: [WinError 10013] An attempt was made to access a socket in a way forbidden by its access permissions


Server error: [WinError 10013] An attempt was made to access a socket in a way forbidden by its access permissions
